# 10 — LightGBM Pre-Ranking (MVP First)

Train **Binary** and **LambdaRank** on Notebook 09's 500 candidates/user; evaluate against original **RRF** on the **same candidate universe**. Time split: Train → Validation (early stopping/model choice) → Test (confirmation only).

**Important:** labels are *weak implicit-feedback labels*, not impressions or CTR; Recall@K denominator is **all eligible future positives** from Notebook 02, not only the retrieved positives. NDCG@K is calculated per eligible evaluation user, including users without a positive among the 500 candidates. PR-AUC is a **candidate classification diagnostic**, not a recommendation ranking metric.

**Run:** VS Code/Jupyter, Conda `torch` kernel; `pip install lightgbm duckdb pyarrow scikit-learn matplotlib`. No need to rerun Notebooks 01–09 if outputs persist.

In [1]:
from pathlib import Path
import json, time, gc, warnings
import numpy as np
import pandas as pd
from IPython.display import display
try:
    import lightgbm as lgb
    from sklearn.metrics import average_precision_score
except ImportError as e:
    raise ImportError('Install missing packages in the active notebook kernel: %pip install lightgbm scikit-learn') from e
try:
    import duckdb
except ImportError as e:
    raise ImportError('Install DuckDB in the active kernel: %pip install duckdb') from e

cwd=Path.cwd().resolve()
ROOT=next((p for p in (cwd,*cwd.parents) if (p/'data'/'processed'/'notebook02.duckdb').is_file()),None)
assert ROOT is not None, 'Cannot find data/processed/notebook02.duckdb; open the project root/notebooks folder.'
OUT=ROOT/'data'/'processed'; REPORT=ROOT/'data'/'reports'; MODEL_DIR=ROOT/'models'
REPORT.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True)
SPLITS=('train','validation','test'); KS=(20,50,100)
paths={s:OUT/f'{s}_ranking_features_top500.parquet' for s in SPLITS}
for s,p in paths.items(): assert p.is_file(),f'Missing {p}: run Notebook 09 first'
manifest_path=REPORT/'notebook09_feature_manifest.json'
assert manifest_path.exists(),'Missing notebook09_feature_manifest.json; run Notebook 09.'
manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
FEATURES=list(manifest['feature_columns']); CATS=list(manifest['categorical_columns'])
assert len(FEATURES)==len(set(FEATURES)) and set(CATS)<=set(FEATURES)
DB=OUT/'notebook02.duckdb'; con=duckdb.connect(str(DB),read_only=True)
print('Root:',ROOT,'LightGBM:',lgb.__version__,'features:',len(FEATURES))

Root: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks LightGBM: 4.6.0 features: 57


## 1. Load and validate the ranking datasets

Preserve **all** users, including users with zero retrieved positives; LightGBM groups must contain exactly 500 consecutive rows per user. User/item IDs and `cutoff_ts` are identifiers, **not** model features.

In [2]:
ID_COLS=['user_id','item_id','cutoff_ts','label','retrieval_rank']
READ_COLS=list(dict.fromkeys(ID_COLS+FEATURES))
data={}; groups={}; eligible={}; audit=[]
for s in SPLITS:
    df=pd.read_parquet(paths[s],columns=READ_COLS)
    missing=set(READ_COLS)-set(df.columns)
    assert not missing,(s,missing)
    df=df.sort_values(['user_id','retrieval_rank'],kind='stable').reset_index(drop=True)
    assert not df[['user_id','item_id']].duplicated().any(),f'{s}: duplicate user-item pairs'
    assert df['label'].isin([0,1]).all(),f'{s}: invalid labels'
    assert df['retrieval_rank'].between(1,500).all(),f'{s}: bad retrieval rank'
    assert not df[FEATURES].isna().any().any(),f'{s}: missing model features'
    group=df.groupby('user_id',sort=False).size().to_numpy(dtype=np.int32)
    assert group.sum()==len(df) and (group==500).all(),f'{s}: expected 500 candidates per user'
    assert (df.groupby('user_id',sort=False)['retrieval_rank'].agg(['min','max'])['min']==1).all()
    assert (df.groupby('user_id',sort=False)['retrieval_rank'].max()==500).all()
    # All Notebook 09 numeric features; categories encoded as category dtype below.
    assert np.isfinite(df[FEATURES].to_numpy(dtype=np.float64,copy=False)).all(),f'{s}: nonfinite features'
    eligible_pairs,eligible_users=con.execute(f'SELECT COUNT(*),COUNT(DISTINCT user_id) FROM {s}_positives').fetchone()
    assert int(eligible_users)==len(group),f'{s}: candidate users and eligible positive users differ'
    assert df['label'].sum()<=eligible_pairs
    eligible[s]={'positive_pairs':int(eligible_pairs),'users':int(eligible_users)}
    audit.append({'split':s,'rows':len(df),'users':len(group),'positives_in_candidates':int(df.label.sum()),'eligible_positives':int(eligible_pairs),'positive_rate':float(df.label.mean()),'min_group':int(group.min()),'max_group':int(group.max())})
    data[s]=df; groups[s]=group
print('Dataset audit:'); display(pd.DataFrame(audit))

Dataset audit:


,split,rows,users,positives_in_candidates,eligible_positives,positive_rate,min_group,max_group
0,train,2055000,4110,2131,9362,0.001037,500,500
1,validation,1828500,3657,1996,9020,0.001092,500,500
2,test,1786000,3572,1785,8214,0.000999,500,500


## 2. LightGBM feature preparation

Use the exact Notebook 09 feature manifest; cast category IDs to categorical with shared category levels across splits (levels contain no target labels). Never include user IDs, item IDs, timestamps, or labels in the model.

In [3]:
# Shared integer category levels are only used as encodings, not for fitting a target-dependent transform.
category_levels={c:sorted(set().union(*(set(data[s][c].astype('int64').unique()) for s in SPLITS))) for c in CATS}
X={}; y={}
for s in SPLITS:
    x=data[s][FEATURES].copy()
    for c in CATS:
        x[c]=pd.Categorical(x[c].astype('int64'),categories=category_levels[c])
    for c in FEATURES:
        if c not in CATS: x[c]=pd.to_numeric(x[c],errors='raise').astype('float32')
    X[s]=x
    y[s]=data[s]['label'].to_numpy(dtype=np.int8)
assert FEATURES==list(X['train'].columns)==list(X['validation'].columns)==list(X['test'].columns)
print('Feature count:',len(FEATURES),'categoricals:',CATS)
print('Train shape:',X['train'].shape)

Feature count: 57 categoricals: ['category_id', 'parent_category_id', 'root_category_id']
Train shape: (2055000, 57)


## 3. Ranking evaluation (Recall, HitRate, NDCG, PR-AUC)

All metrics computed **over every eligible evaluation user**. For a user with zero retrieved positives, NDCG is zero. Ideal DCG uses the number of *all eligible future positives* for that user (capped at K), rather than only retrieved positives; this makes RRF/LightGBM NDCG comparable and acknowledges retrieval misses. We also report candidate-normalized NDCG separately if useful later.

In [4]:
# Preload eligible-positive counts for each evaluation user from Notebook 02.
pos_counts={}
for s in SPLITS:
    t=con.execute(f'SELECT user_id,COUNT(*) AS n FROM {s}_positives GROUP BY user_id').df()
    pos_counts[s]=dict(zip(t.user_id.astype('int64'),t.n.astype('int32')))
    assert len(pos_counts[s])==eligible[s]['users']

def evaluate_predictions(s,method,scores,ks=KS):
    df=data[s]; scores=np.asarray(scores,dtype=np.float64)
    assert len(scores)==len(df) and np.isfinite(scores).all()
    # deterministic tie-break using original retrieval_rank then item_id
    pred=pd.DataFrame({'user_id':df.user_id.to_numpy(),'item_id':df.item_id.to_numpy(),
        'label':y[s], 'score':scores,'original_rank':df.retrieval_rank.to_numpy()})
    pred=pred.sort_values(['user_id','score','original_rank','item_id'],ascending=[True,False,True,True],kind='stable')
    pred['model_rank']=pred.groupby('user_id',sort=False).cumcount()+1
    user_id=pred['user_id'].to_numpy(); labels=pred['label'].to_numpy(dtype=np.int8)
    ranks=pred['model_rank'].to_numpy(); user_starts=np.r_[0,np.flatnonzero(user_id[1:]!=user_id[:-1])+1]
    user_ids=user_id[user_starts]
    positives=np.asarray([pos_counts[s][int(u)] for u in user_ids],dtype=np.int32)
    assert len(user_ids)==eligible[s]['users'] and np.all(positives>0)
    results=[]
    for k in ks:
        mask=ranks<=k
        hits=int(labels[mask].sum())
        user_hit=np.add.reduceat((labels*(ranks<=k)).astype(np.int32),user_starts)
        # DCG = sum 1/log2(rank+1) over positive labels within top K
        gains=np.where(mask,labels/np.log2(ranks+1),0.0)
        dcg=np.add.reduceat(gains,user_starts)
        # IDCG based on *all eligible future positives*, not just 500 candidates
        discounts=1/np.log2(np.arange(2,k+2,dtype=np.float64))
        cumdiscount=np.r_[0.,np.cumsum(discounts)]
        idcg=cumdiscount[np.minimum(positives,k)]
        ndcg=float(np.mean(dcg/idcg))
        results.append({'split':s,'method':method,'k':int(k),
            'hits':hits,'eligible_positive_pairs':eligible[s]['positive_pairs'],
            'recall_micro':hits/eligible[s]['positive_pairs'],
            'hit_rate':float(np.mean(user_hit>0)), 'ndcg':ndcg,
            'user_coverage':len(user_ids)/eligible[s]['users']})
    ap=float(average_precision_score(y[s],scores)) if y[s].sum()>0 else float('nan')
    return results,ap,pred

baseline_metrics=[]; baseline_ap={}
for s in SPLITS:
    # Higher original RRF rank = lower priority. Use negative rank, which exactly reproduces baseline.
    rows,ap,_=evaluate_predictions(s,'rrf_baseline',-data[s].retrieval_rank.to_numpy())
    baseline_metrics.extend(rows);baseline_ap[s]=ap
print('RRF baseline:');display(pd.DataFrame(baseline_metrics).query("split=='validation'"))

RRF baseline:


,split,method,k,hits,eligible_positive_pairs,recall_micro,hit_rate,ndcg,user_coverage
3,validation,rrf_baseline,20,615,9020,0.068182,0.131255,0.045277,1.0
4,validation,rrf_baseline,50,927,9020,0.102772,0.178015,0.054554,1.0
5,validation,rrf_baseline,100,1185,9020,0.131375,0.218485,0.060974,1.0


## 4. Binary classification baseline

A compact tree ensemble, early stopping by **Validation average precision** (PR-AUC), with a second, unbiased user-ranking evaluation below. `is_unbalance`/class weights are intentionally omitted: our goal is a ranking baseline and score ordering, not calibrated probabilities.

In [5]:
SEED=42
binary_params=dict(objective='binary',metric='average_precision',learning_rate=0.05,
    num_leaves=31,max_depth=-1,min_data_in_leaf=100,feature_fraction=0.85,
    bagging_fraction=0.85,bagging_freq=1,lambda_l2=2.0,verbosity=-1,
    num_threads=4,seed=SEED,feature_fraction_seed=SEED,bagging_seed=SEED)
train_bin=lgb.Dataset(X['train'],label=y['train'],categorical_feature=CATS,free_raw_data=False)
val_bin=lgb.Dataset(X['validation'],label=y['validation'],reference=train_bin,categorical_feature=CATS,free_raw_data=False)
t0=time.perf_counter()
binary_model=lgb.train(binary_params,train_bin,num_boost_round=500,
    valid_sets=[val_bin],valid_names=['validation'],
    callbacks=[lgb.early_stopping(stopping_rounds=40,verbose=False),lgb.log_evaluation(period=50)])
binary_seconds=time.perf_counter()-t0
print('Binary best iteration:',binary_model.best_iteration,'train seconds:',round(binary_seconds,2))

[50]	validation's average_precision: 0.0166517
Binary best iteration: 19 train seconds: 10.46


## 5. LambdaRank (group-aware main model)

LightGBM ranking objective optimizes within-user ordering. Each group contains exactly 500 consecutive candidates. Users with no positive among the 500 remain included (they provide no ranking gradients, but are essential to unbiased coverage/metric denominators). Early stopping uses **Validation NDCG@20**, not Test.

In [6]:
rank_params=dict(objective='lambdarank',metric='ndcg',ndcg_eval_at=[20,50,100],
    label_gain=[0,1],learning_rate=0.05,num_leaves=31,min_data_in_leaf=100,
    feature_fraction=0.85,bagging_fraction=0.85,bagging_freq=1,lambda_l2=2.0,
    lambdarank_truncation_level=30,verbosity=-1,num_threads=4,
    seed=SEED,feature_fraction_seed=SEED,bagging_seed=SEED)
train_rank=lgb.Dataset(X['train'],label=y['train'],group=groups['train'],categorical_feature=CATS,free_raw_data=False)
val_rank=lgb.Dataset(X['validation'],label=y['validation'],group=groups['validation'],reference=train_rank,categorical_feature=CATS,free_raw_data=False)
assert np.array_equal(np.repeat(np.arange(len(groups['train'])),groups['train']),
                      pd.factorize(data['train'].user_id,sort=False)[0])
t0=time.perf_counter()
rank_model=lgb.train(rank_params,train_rank,num_boost_round=500,
    valid_sets=[val_rank],valid_names=['validation'],
    callbacks=[lgb.early_stopping(stopping_rounds=40,first_metric_only=True,verbose=False),
               lgb.log_evaluation(period=50)])
rank_seconds=time.perf_counter()-t0
print('LambdaRank best iteration:',rank_model.best_iteration,'train seconds:',round(rank_seconds,2))

[50]	validation's ndcg@20: 0.72621	validation's ndcg@50: 0.737812	validation's ndcg@100: 0.746441
LambdaRank best iteration: 53 train seconds: 13.27


## 6. Compare RRF vs Binary vs LambdaRank

Do model selection **only on Validation**; keep Test untouched until the choice is fixed. Candidate Recall@500 cannot exceed the Notebook 08 pool ceiling without new retrieval.

In [7]:
MODELS={'binary':binary_model,'lambdarank':rank_model}
PRED={}; METRICS=list(baseline_metrics); AP=[]
for s in SPLITS:
    AP.append({'split':s,'method':'rrf_baseline','pr_auc':baseline_ap[s]})
for name,model in MODELS.items():
    for s in ('train','validation'):
        scores=model.predict(X[s],num_iteration=model.best_iteration)
        PRED[(s,name)]=scores
        rows,ap,_=evaluate_predictions(s,name,scores)
        METRICS.extend(rows);AP.append({'split':s,'method':name,'pr_auc':ap})
metrics_df=pd.DataFrame(METRICS);ap_df=pd.DataFrame(AP)
print('Validation ranking metrics:')
display(metrics_df.query("split=='validation'").pivot(index='method',columns='k',values=['recall_micro','hit_rate','ndcg']).round(5))
print('Validation PR-AUC:');display(ap_df.query("split=='validation'"))
# Pick main model by our independently computed full-eligible-user NDCG@20.
validation_ndcg=metrics_df.query("split=='validation' and k==20").set_index('method')['ndcg']
chosen=max(('binary','lambdarank'),key=lambda name:float(validation_ndcg[name]))
print('Selected LightGBM model by validation NDCG@20:',chosen)
print('Validation NDCG@20:',{k:round(float(v),6) for k,v in validation_ndcg.items()})

Validation ranking metrics:


recall_micro                   hit_rate                    \
k                     20       50       100      20       50       100   
method                                                                   
binary            0.07106  0.10920  0.14246  0.13071  0.18430  0.23052   
lambdarank        0.06763  0.10399  0.13625  0.12961  0.18321  0.22313   
rrf_baseline      0.06818  0.10277  0.13137  0.13126  0.17801  0.21849   

                 ndcg                    
k                 20       50       100  
method                                   
binary        0.05085  0.06153  0.06937  
lambdarank    0.04886  0.05912  0.06603  
rrf_baseline  0.04528  0.05455  0.06097

Validation PR-AUC:


,split,method,pr_auc
1,validation,rrf_baseline,0.006342
4,validation,binary,0.017820
6,validation,lambdarank,0.011969


Selected LightGBM model by validation NDCG@20: binary
Validation NDCG@20: {'rrf_baseline': 0.045277, 'binary': 0.050853, 'lambdarank': 0.048861}


## 7. One-time Test evaluation (after model selection)

Test is **never** used for hyperparameter tuning, early stopping, or model selection. Both trained models are evaluated here for reporting; `chosen` remains fixed from Validation.

In [8]:
for name,model in MODELS.items():
    scores=model.predict(X['test'],num_iteration=model.best_iteration)
    PRED[('test',name)]=scores
    rows,ap,_=evaluate_predictions('test',name,scores)
    METRICS.extend(rows);AP.append({'split':'test','method':name,'pr_auc':ap})
metrics_df=pd.DataFrame(METRICS);ap_df=pd.DataFrame(AP)
print('Test results (confirmatory):')
display(metrics_df.query("split=='test'").pivot(index='method',columns='k',values=['recall_micro','hit_rate','ndcg']).round(5))
display(ap_df.query("split=='test'"))

Test results (confirmatory):


recall_micro                   hit_rate                    \
k                     20       50       100      20       50       100   
method                                                                   
binary            0.07256  0.10713  0.14244  0.13158  0.17945  0.22676   
lambdarank        0.06489  0.10019  0.13550  0.12010  0.16993  0.21809   
rrf_baseline      0.06672  0.09910  0.12978  0.12374  0.17105  0.21221   

                 ndcg                    
k                 20       50       100  
method                                   
binary        0.05400  0.06334  0.07109  
lambdarank    0.04683  0.05607  0.06380  
rrf_baseline  0.04718  0.05590  0.06252

,split,method,pr_auc
2,test,rrf_baseline,0.006218
7,test,binary,0.014597
8,test,lambdarank,0.009456


## 8. Feature importance and outputs

Exports predictions for all three methods; each model's prediction file contains the original candidate IDs, label, original RRF rank, model score, and model rank. The model files can be loaded with `lgb.Booster(model_file=...)`.

In [9]:
importance=[]
for name,model in MODELS.items():
    for typ in ('gain','split'):
        vals=model.feature_importance(importance_type=typ)
        importance.extend({'model':name,'importance_type':typ,'feature':f,'importance':float(v)} for f,v in zip(FEATURES,vals))
importance_df=pd.DataFrame(importance)
for name,model in MODELS.items():
    dest=MODEL_DIR/f'notebook10_lightgbm_{name}.txt'
    model.save_model(str(dest))
    print('Saved model:',dest)
print('Top LambdaRank gain features:')
display(importance_df.query("model=='lambdarank' and importance_type=='gain'").sort_values('importance',ascending=False).head(20))

Saved model: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\models\notebook10_lightgbm_binary.txt
Saved model: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\models\notebook10_lightgbm_lambdarank.txt
Top LambdaRank gain features:


,model,importance_type,feature,importance
155,lambdarank,gain,category_id,31611.027823
114,lambdarank,gain,retrieval_rank,21676.712838
115,lambdarank,gain,rrf_score,8223.339515
167,lambdarank,gain,user_category_affinity,3866.825849
156,lambdarank,gain,parent_category_id,3164.403606
150,lambdarank,gain,item_recency_days,2361.945099
166,lambdarank,gain,category_30d_events,1898.395202
128,lambdarank,gain,category_rank,1509.400798
140,lambdarank,gain,user_recency_days,1327.206203
124,lambdarank,gain,cf_rank,966.866104


In [10]:
prediction_exports=[]
for s in SPLITS:
    for method in ('rrf_baseline','binary','lambdarank'):
        scores=(-data[s].retrieval_rank.to_numpy() if method=='rrf_baseline' else PRED[(s,method)])
        _,_,pred=evaluate_predictions(s,method,scores)
        dest=OUT/f'{s}_lightgbm_{method}_top500.parquet'
        pred.to_parquet(dest,index=False,compression='zstd')
        prediction_exports.append({'split':s,'method':method,'path':dest.name,'rows':len(pred),'size_mb':round(dest.stat().st_size/1048576,2)})
metrics_df.to_csv(REPORT/'notebook10_ranking_metrics.csv',index=False)
ap_df.to_csv(REPORT/'notebook10_pr_auc.csv',index=False)
importance_df.to_csv(REPORT/'notebook10_feature_importance.csv',index=False)
pd.DataFrame(audit).to_csv(REPORT/'notebook10_data_audit.csv',index=False)
print('Prediction files:');display(pd.DataFrame(prediction_exports))

Prediction files:


,split,method,path,rows,size_mb
0,train,rrf_baseline,train_lightgbm_rrf_baseline_top500.parquet,2055000,2.80
1,train,binary,train_lightgbm_binary_top500.parquet,2055000,6.95
2,train,lambdarank,train_lightgbm_lambdarank_top500.parquet,2055000,15.15
3,validation,rrf_baseline,validation_lightgbm_rrf_baseline_top500.parquet,1828500,2.49
4,validation,binary,validation_lightgbm_binary_top500.parquet,1828500,6.28
5,validation,lambdarank,validation_lightgbm_lambdarank_top500.parquet,1828500,13.52
6,test,rrf_baseline,test_lightgbm_rrf_baseline_top500.parquet,1786000,2.41
7,test,binary,test_lightgbm_binary_top500.parquet,1786000,5.89
8,test,lambdarank,test_lightgbm_lambdarank_top500.parquet,1786000,12.71


## 9. Sanity checks and machine-readable report

In [11]:
# Critical checks: RRF baseline must reproduce Notebook 08's exact Recall@K.
reference_path=REPORT/'notebook08_fusion_metrics.csv'
if reference_path.is_file():
    ref=pd.read_csv(reference_path)
    for s in SPLITS:
        for k in KS:
            ours=metrics_df.query('split==@s and method=="rrf_baseline" and k==@k').iloc[0]
            previous=ref.query('split==@s and k==@k')
            if 'method' in previous.columns: previous=previous[previous.method=='fusion_rrf']
            assert len(previous)==1,(s,k,'ambiguous Notebook 08 reference')
            assert int(ours.hits)==int(previous.iloc[0].hits),(s,k,'RRF mismatch')
    print('RRF Recall@K matches Notebook 08 exactly.')
else:
    print('Notebook 08 metrics CSV not found; skipping cross-notebook baseline equality check.')
assert len(prediction_exports)==9
assert all(m['user_coverage']==1.0 for m in METRICS)
assert set(metrics_df.method)=={'rrf_baseline','binary','lambdarank'}
assert chosen in MODELS
assert all(np.isfinite(ap_df.pr_auc))
report={'notebook':'10_lightgbm_pre_ranking','objective':'Binary baseline + LambdaRank group-aware ranking',
    'input_files':{s:paths[s].name for s in SPLITS},'feature_count':len(FEATURES),
    'categorical_features':CATS,'features':FEATURES,'ks':list(KS),
    'label_definition':'eligible future positive among 500 candidates; other candidates weak negatives',
    'metric_definition':{'recall_micro':'topK positive hits / all eligible future positive pairs from Notebook 02',
      'hit_rate':'fraction of all eligible evaluation users with >=1 topK hit',
      'ndcg':'mean user DCG@K / ideal DCG@K from all eligible future positives; zero for misses',
      'pr_auc':'average precision over the fixed candidate pool; not a CTR estimate'},
    'training':{'binary_best_iteration':int(binary_model.best_iteration),
      'lambdarank_best_iteration':int(rank_model.best_iteration),
      'binary_seconds':round(binary_seconds,3),'lambdarank_seconds':round(rank_seconds,3),
      'model_selection':'highest validation full-eligible-user NDCG@20 among LightGBM models',
      'selected_model':chosen,'test_not_used_for_selection':True},
    'data_audit':audit,'ranking_metrics':metrics_df.to_dict(orient='records'),
    'pr_auc':ap_df.to_dict(orient='records'),'prediction_exports':prediction_exports,
    'model_files':{name:f'notebook10_lightgbm_{name}.txt' for name in MODELS},
    'limitations':['weak implicit negatives; no impression logs','candidate ranking cannot recover positives missing from retrieval',
      'NDCG computed on eligible positive universe, not only candidate positives',
      'PIT is split-level (not event-level) as in Notebook 09','no real online latency or causal CTR/CVR']}
report_path=REPORT/'notebook10_lightgbm_report.json'
report_path.write_text(json.dumps(report,indent=2,allow_nan=False),encoding='utf-8')
print('Saved report:',report_path)
print('Selected model:',chosen)
print('Next: upload notebook10_lightgbm_report.json for analysis and Notebook 11 DeepFM.')

RRF Recall@K matches Notebook 08 exactly.
Saved report: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\reports\notebook10_lightgbm_report.json
Selected model: binary
Next: upload notebook10_lightgbm_report.json for analysis and Notebook 11 DeepFM.
